# 6 

In [2]:
%pip install llama-index llama-index-llms-openai-like
import os
import pandas as pd
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.embeddings import MockEmbedding
from llama_index.llms.openai_like import OpenAILike
# Groq API key
os.environ["GROQ_API_KEY"] = "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"
# Use Groq with OpenAI-compatible endpoint
Settings.llm = OpenAILike(
    model="openai/gpt-oss-20b",
    api_base="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    is_chat_model=True,
    is_function_calling_model=False
)
# Simple embedding for classroom demo
Settings.embed_model = MockEmbedding(embed_dim=384)
# Sales report data
sales_df = pd.DataFrame({
    "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
    "Product": ["Laptop", "Mobile", "Tablet", "Laptop", "Mobile"],
    "Region": ["South", "North", "East", "West", "South"],
    "Revenue": [50000, 40000, 25000, 60000, 55000],
    "Profit": [10000, 8000, 4000, 12000, 11000]
})
display(sales_df)
# Convert table into document
sales_text = sales_df.to_string(index=False)
document = Document(
    text=f"""
Sales Report:
{sales_text}
"""
)
# Create LlamaIndex index
index = VectorStoreIndex.from_documents([document])
# Create query engine
query_engine = index.as_query_engine()
# Ask question
response = query_engine.query(
    "Which product generated the highest total revenue? Show calculation."
)
print(response)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


,Month,Product,Region,Revenue,Profit
0,Jan,Laptop,South,50000,10000
1,Feb,Mobile,North,40000,8000
2,Mar,Tablet,East,25000,4000
3,Apr,Laptop,West,60000,12000
4,May,Mobile,South,55000,11000


**Laptop** generated the highest total revenue.  

Calculation:  
- January Laptop: $50,000  
- April Laptop: $60,000  

Total Laptop revenue = $50,000 + $60,000 = **$110,000**.


# 7

In [5]:
!pip install -U cohere numpy pandas

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
import cohere
import numpy as np
import pandas as pd
# 1. Set Cohere API Key
os.environ["COHERE_API_KEY"] = "y3roeFf01TazGMVWV2Aj8MMHmFOT2j6qq4K98EOo"
co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
# 2. Market research knowledge base
documents = [
    "Electric vehicles are growing rapidly due to rising fuel prices, government incentives, and environmental awareness.",
    "The Indian EV market is driven by two-wheelers, three-wheelers, and urban mobility demand.",
    "Major challenges in EV adoption include charging infrastructure, battery cost, range anxiety, and supply chain limitations.",
    "Key competitors in the EV market include Tata Motors, Mahindra Electric, Ola Electric, Ather Energy, and MG Motor.",
    "Customers prefer EVs because of low running cost, reduced emissions, and government subsidy benefits.",
    "Battery swapping and fast charging networks are emerging as important opportunities in the EV ecosystem.",
    "The market is expected to expand with support from renewable energy integration and smart mobility policies."
]
# 3. Create document embeddings
doc_embed_response = co.embed(
    model="embed-v4.0",
    texts=documents,
    input_type="search_document",
    embedding_types=["float"]
)
doc_embeddings = np.array(doc_embed_response.embeddings.float)
# 4. Similarity search function
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
def retrieve_documents(query, top_k=5):
    query_embed_response = co.embed(
        model="embed-v4.0",
        texts=[query],
        input_type="search_query",
        embedding_types=["float"]
    )
    query_embedding = np.array(query_embed_response.embeddings.float[0])
    scores = [
        cosine_similarity(query_embedding, doc_embedding)
        for doc_embedding in doc_embeddings
    ]
    ranked_results = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return ranked_results[:top_k]
# 5. Rerank with Cohere Rerank
def rerank_documents(query, retrieved_docs, top_n=3):
    docs_only = [doc for doc, score in retrieved_docs]
    rerank_response = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=docs_only,
        top_n=top_n
    )
    reranked_docs = [
        docs_only[result.index]
        for result in rerank_response.results
    ]
    return reranked_docs
# 6. Market Research Agent
def market_research_agent(query):
    retrieved_docs = retrieve_documents(query, top_k=5)
    final_docs = rerank_documents(query, retrieved_docs, top_n=3)

    context = "\n".join([f"- {doc}" for doc in final_docs])

    prompt = f"""
You are a professional Market Research Agent.

User Query:
{query}

Relevant Market Information:
{context}

Prepare a structured market research report with:
1. Market Overview
2. Key Drivers
3. Challenges
4. Competitor Landscape
5. Opportunities
6. Final Recommendation
"""

    response = co.chat(
        model="command-a-03-2025",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content[0].text

# 7. Run the agent
query = "Prepare a market research report on electric vehicles in India"
answer = market_research_agent(query)

print(answer)